# SFT 监督微调原理与实战

## 什么是 SFT（Supervised Fine-Tuning）？

SFT（监督微调）是在预训练大语言模型的基础上，使用**标注好的指令-回复数据对**进行有监督训练，让模型学会按照特定格式和风格回答问题。

### 为什么需要微调？

| 动机 | 说明 | B站场景示例 |
|------|------|-------------|
| **领域适配** | 让模型掌握特定领域知识 | 学习B站广告文案的行业术语和投放逻辑 |
| **风格控制** | 统一输出的语气和风格 | 生成符合B站年轻用户调性的活泼文案 |
| **格式控制** | 输出结构化、规范化内容 | 严格按照标题+卖点+CTA的广告文案格式 |
| **成本优化** | 小模型替代大模型完成特定任务 | 用微调后的7B模型替代GPT-4生成广告文案 |

### LLM 训练三阶段

```
预训练（Pretraining）→ SFT（监督微调）→ RLHF（人类反馈强化学习）
     海量文本             指令数据对           人类偏好排序
     学习语言能力          学习遵循指令         学习人类偏好
```

In [ ]:
import json

# ============================================================
# SFT 数据格式：instruction / input / output（Alpaca格式）
# 这是最常用的微调数据格式之一
# ============================================================

sft_samples = [
    {
        "instruction": "根据以下产品信息，撰写一条B站信息流广告文案。",
        "input": "产品：某品牌蓝牙耳机，特点：主动降噪、续航40小时、适合学生党",
        "output": "🎧 上课摸鱼神器来了！主动降噪让你沉浸学习，40小时续航一周不用充，学生党闭眼入！点击领取专属优惠👇"
    },
    {
        "instruction": "根据以下产品信息，撰写一条B站信息流广告文案。",
        "input": "产品：编程网课，特点：Python入门到实战、AI方向、适合零基础",
        "output": "零基础转码上岸！从Python小白到AI大佬，这门课带你卷赢同龄人💪 限时特惠，评论区领券！"
    },
    {
        "instruction": "根据以下产品信息，撰写一条B站信息流广告文案。",
        "input": "产品：护肤套装，特点：烟酰胺精华、控油保湿、男女通用",
        "output": "熬夜追番脸垮了？烟酰胺精华拯救你的暗沉肌！控油保湿一步到位，男女都能用～ 戳链接get同款✨"
    },
    {
        "instruction": "根据以下产品信息，撰写一条B站信息流广告文案。",
        "input": "产品：机械键盘，特点：RGB灯效、红轴静音、客制化",
        "output": "码字手感天花板🔥 红轴静音不扰室友，RGB灯效拉满氛围感，客制化玩家必入！限量配色抢先看👀"
    },
    {
        "instruction": "根据以下产品信息，撰写一条B站信息流广告文案。",
        "input": "产品：速溶咖啡，特点：冷萃工艺、0蔗糖、便携装",
        "output": "DDL战士续命神器☕ 冷萃工艺还原现磨口感，0蔗糖喝了不怕胖！随身带一包，随时回血～ 囤货链接在这里！"
    }
]

# 展示数据格式
print("=" * 60)
print("B站广告文案 SFT 训练数据示例")
print("=" * 60)
for i, sample in enumerate(sft_samples[:3]):
    print(f"\n--- 样本 {i+1} ---")
    print(json.dumps(sample, ensure_ascii=False, indent=2))

# 保存为 JSONL 格式（微调标准格式）
print(f"\n共 {len(sft_samples)} 条训练样本")
print("实际项目中通常需要 1000~10000 条高质量数据")

In [ ]:
# ============================================================
# SFT 训练流程（概念伪代码）
# ============================================================

print("SFT 监督微调训练流程")
print("=" * 60)

training_pipeline = """
Step 1: 加载预训练模型
   model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-7B")
   tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B")

Step 2: 准备数据 → 转换为模型输入格式
   prompt_template = '''<|im_start|>system
   你是B站广告文案生成助手。<|im_end|>
   <|im_start|>user
   {instruction}\n{input}<|im_end|>
   <|im_start|>assistant
   {output}<|im_end|>'''

Step 3: 配置训练参数
   - learning_rate: 2e-5（SFT常用范围：1e-5 ~ 5e-5）
   - epochs: 3（通常 2~5 轮足够）
   - batch_size: 4（受显存限制）
   - max_seq_length: 512（根据数据长度调整）

Step 4: 训练
   trainer.train()  # 在标注数据上进行有监督训练

Step 5: 评估 & 推理
   - 自动指标：loss、perplexity
   - 人工评估：文案质量、相关性、吸引力
   - A/B测试：对比微调前后的广告点击率
"""
print(training_pipeline)

In [ ]:
# ============================================================
# HuggingFace Trainer 实战代码
# ============================================================

try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
        Trainer,
        DataCollatorForLanguageModeling,
    )
    from datasets import Dataset

    # --- 1. 加载模型和分词器 ---
    model_name = "Qwen/Qwen2-1.5B"  # 演示用小模型
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

    # --- 2. 构造数据集 ---
    def format_sample(sample):
        """将 instruction/input/output 转换为模型训练格式"""
        text = (
            f"<|im_start|>user\n{sample['instruction']}\n{sample['input']}<|im_end|>\n"
            f"<|im_start|>assistant\n{sample['output']}<|im_end|>"
        )
        return tokenizer(text, truncation=True, max_length=512, padding="max_length")

    dataset = Dataset.from_list(sft_samples).map(format_sample)

    # --- 3. 训练参数 ---
    training_args = TrainingArguments(
        output_dir="./bilibili_ad_sft",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        learning_rate=2e-5,
        warmup_steps=10,
        logging_steps=5,
        save_strategy="epoch",
        fp16=True,  # 混合精度训练
    )

    # --- 4. 初始化 Trainer ---
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("Trainer 初始化成功！")
    print(f"模型参数量: {model.num_parameters() / 1e9:.1f}B")
    print(f"训练样本数: {len(dataset)}")
    # trainer.train()  # 取消注释即可开始训练

except ImportError:
    print("提示：需要安装 transformers 和 datasets 库")
    print("pip install transformers datasets torch")
    print("\n以上代码展示了完整的 HuggingFace Trainer SFT 流程：")
    print("1. 加载预训练模型 → 2. 格式化数据 → 3. 配置训练参数 → 4. 启动训练")

## SFT vs Pretraining vs RLHF 对比

| 维度 | Pretraining（预训练） | SFT（监督微调） | RLHF（人类反馈强化学习） |
|------|----------------------|----------------|------------------------|
| **目标** | 学习语言规律 | 学习遵循指令 | 学习人类偏好 |
| **数据** | 万亿token无标注文本 | 万~十万条指令对 | 人类偏好排序数据 |
| **成本** | 数百万美元 | 数百~数千美元 | 数千~数万美元 |
| **训练时长** | 数周~数月 | 数小时~数天 | 数天~数周 |
| **B站场景** | 不涉及（使用开源模型） | 训练广告文案生成 | 基于投放效果优化文案质量 |

---

## 面试高频问题

**Q: SFT 和 Pretraining 的本质区别是什么？**
> Pretraining 是无监督的 next-token prediction，学的是语言能力；SFT 是有监督训练，让模型学会「按指令办事」。数据格式从纯文本变成了 instruction-response 对。

**Q: SFT 需要多少数据？**
> 取决于任务复杂度。单一任务（如B站广告文案）1000~5000 条高质量数据通常足够；通用能力提升需要 10 万+条。数据质量远比数量重要。

**Q: SFT 会导致灾难性遗忘吗？**
> 会。微调后模型可能在通用任务上性能下降。缓解方法：(1) 混入通用数据 (2) 降低学习率 (3) 使用 LoRA 等参数高效方法（下节课讲解）。